In [27]:


using Gen
using Plots
using Statistics
using Distributions 



function compute_rhat(chains)
    m = length(chains)   # Number of chains
    n = length(chains[1])  # Number of samples per chain
   
    # Compute chain means
    chain_means = [mean(chain) for chain in chains]
    grand_mean = mean(chain_means)

    # Compute within-chain variance W
    W = mean([var(chain, corrected=true) for chain in chains])  # corrected=true uses n-1 in denominator
    #println("W", W)
    # Compute between-chain variance B
    B = (n / (m - 1)) * sum((chain_mean - grand_mean)^2 for chain_mean in chain_means)
    #println("B", B)
    # Compute potential scale reduction factor (R-hat)
    var_hat = ((n - 1) / n) * W + (B / n)
    r_hat = sqrt(var_hat / W)
    #println(var_hat)
    #println(r_hat)
    return r_hat
end





# Define the model with dynamic eta sampling
@gen function eight_school_model(sigma)
    mu ~ normal(0, 1)     # Sample mu from a normal distribution
    tau  ~ normal(2, 5)   # Sample tau from a Half-Cauchy distribution
    
    # Dynamic eta values sampled from normal distributions
    list_of_Eta = [{(:eta, i)} ~ normal(0, 1) for i=1:length(sigma)]
    
    for i in 1:length(sigma)  # Loop over 5 iterations
        # Calculate theta based on mu, tau, and eta
        theta = mu + tau * list_of_Eta[i]
        
        # Sample obs from a normal distribution with mean theta and standard deviation sigma[i]
         {(:y, i)} ~ normal(theta, sigma[i])
    end

    
    
    
end

function multi_variable_metropolis(trace, model, sigma, observations, eps)
    # Extract current values of parameters
    mu_current = get_choices(trace)[:mu]
    tau_current = get_choices(trace)[:tau]
    eta_current = [get_choices(trace)[(:eta, i)] for i in 1:length(sigma)]
    
    
    # Propose new values for mu, tau, and eta
    mu_proposed = mu_current + eps * randn()
    tau_proposed = tau_current +  eps *  randn()
    eta_proposed = [eta_current[i] + eps*randn() for i in 1:length(sigma)]
    
     # Create a temporary choice map with the proposed values
    temp_cm = choicemap(
        (:mu => mu_proposed),
        (:tau => tau_proposed)
    )
   
    # Add proposed eta values to the choice map
    for i in 1:length(sigma)
        
        temp_cm[(:eta, i)] = eta_proposed[i]
        # print("hoi") 
    end

    # Use Gen.update to get the updated trace after applying the proposed values
    (proposed_trace, _, _) = Gen.update(
        trace,                # Current trace                # Generative model
        (sigma,), (),            # Arguments for the generative function
        temp_cm               # Temporary choice map with proposed values
    )

    ###
    current_score = get_score(trace)
    #print(current_score)
    
    proposed_score = get_score(proposed_trace)
    # print(proposed_score)
    # Compute the acceptance ratio using the gradients
    acceptance_ratio = min(1.0, exp(proposed_score - current_score))
    # print(acceptance_ratio)
    # Accept or reject based on the acceptance ratio
    if rand() < acceptance_ratio
        return (proposed_trace, 1)
    else
        return (trace, 0)  # Keep the current state if not accepted
    end
end

function do_inference(model, sigma, y_obs, num_iters, eps)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Store sampled values at each iteration
    for _ in 1:num_iters
        (trace, accepted_this_iter) = multi_variable_metropolis(trace, model, sigma, observations, eps)
        accepted += accepted_this_iter
        
        # Store samples
        final_choices = get_choices(trace)
        push!(mu_samples, final_choices[:mu])
        push!(tau_samples, final_choices[:tau])
    end

    acceptance_rate = accepted / num_iters  # Compute acceptance rate

    return (mu_samples, tau_samples, acceptance_rate)
end





sigma = [15, 10, 16, 11, 9, 11, 10, 18]
y_obs = [28, 8, -3, 7, -1, 1, 18, 12]
#y_obs = [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
num_iters = 2000  # Number of iterations for Metropolis-Hastings
eps = 0.1         # Step size for proposals
num_chains = 20
for eps in [0.1, 0.5, 0.9, 1, 1.1, 1.2, 1.5, 1.7, 2, 7]
    chains_mu = []
    chains_tau = []
    acceptance_rates = []
    for _ in 1:num_chains
        mu_samples, tau_samples, acc = do_inference(eight_school_model, sigma, y_obs, num_iters, eps)
        push!(chains_mu, mu_samples)
        push!(chains_tau, tau_samples)
        
        push!(acceptance_rates, acc)
    end
    println("Eps: ", eps)
    rhat_mu = compute_rhat(chains_mu)
    rhat_tau = compute_rhat(chains_tau)
    println("R-hat for mu: ", rhat_mu)
    println("R-hat for tau: ", rhat_tau)
    means_per_chain = [mean(chain) for chain in chains_mu]
    println("Mean of mu: ", mean(means_per_chain))
    means_per_chain_tau = [mean(chain) for chain in chains_tau]
    println("Mean of tau: ", mean(means_per_chain_tau))
    println("acceptance rate: ", mean(acceptance_rates))
    
end



Eps: 0.1
R-hat for mu: 1.0915529052924096
R-hat for tau: 3.448417903869654
Mean of mu: 0.2185980641777098
2.4780188363499605
acceptance rate: 0.8668500000000001
0.5: 
R-hat for mu: 1.004849345839416
R-hat for tau: 1.1834067900844365
Mean of mu: 0.38749068920563595
Mean of tau: 1.4289270049672156
acceptance rate: 0.4427
0.9: 
R-hat for mu: 1.0068654441087685
R-hat for tau: 1.1387103816028048
Mean of mu: 0.3541789546121483
Mean of tau: 2.029388079361696
acceptance rate: 0.18094999999999997
1.0: 
R-hat for mu: 1.0078436985494774
R-hat for tau: 1.1702707122048381
Mean of mu: 0.4348284119590423
Mean of tau: 1.516142650512553
acceptance rate: 0.14385
Eps: 1.1
R-hat for mu: 1.0086164121648566
R-hat for tau: 1.2250423259293028
Mean of mu: 0.3972722900766593
Mean of tau: 2.1009925945074395
acceptance rate: 0.1101
1.2: 
R-hat for mu: 1.01337209212032
R-hat for tau: 1.480462875892361
Mean of mu: 0.3729677253002638
Mean of tau: 1.2189072289525418
acceptance rate: 0.0864
Eps: 1.5
R-hat for mu: 1.02